<a href="https://colab.research.google.com/github/LogBlast/Analyseur-de-texte-multilingue/blob/main/projet_cuda_bruteforce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projet CUDA : Brute-Force d'un chiffrement XOR

On a récupéré un fichier chiffré par XOR. Le chiffrement XOR fonctionne comme ça :
- Chaque octet du texte est XORé avec un octet de la clé (en boucle)
- Pour déchiffrer, il suffit de XORer à nouveau avec la même clé

On va explorer trois approches pour retrouver la clé :
1. **Approche intelligente (CPU)** : on connaît le texte original en entier, on résout la clé lettre par lettre
2. **Brute-force séquentiel (CPU)** : on ne connaît qu'un préfixe, on teste toutes les combinaisons une par une
3. **Brute-force parallèle (GPU)** : même problème que (2), mais 11,8 millions de threads en simultané

## Cellule 1 — Mise en place

In [ ]:
import numpy as np
from numba import cuda
import time
import math

# le message qu'on simule avoir intercepté
texte_original    = "BEGIN: rapport_confidentiel_2024.txt"
cle_secrete_texte = "STORM"

# conversion en listes d'entiers ASCII (chaque lettre devient son code numérique)
texte_ascii = [ord(c) for c in texte_original]
cle_ascii   = [ord(c) for c in cle_secrete_texte]

# chiffrement XOR : chaque caractère du texte est combiné avec un caractère de la clé
ciphertext_str = ""
for i in range(len(texte_ascii)):
    ciphertext_str += chr(texte_ascii[i] ^ cle_ascii[i % len(cle_ascii)])

# version numpy du texte chiffré, nécessaire pour l'envoyer au GPU plus tard
ciphertext_np = np.array([ord(c) for c in ciphertext_str], dtype=np.uint8)

# le préfixe connu qu'on va utiliser pour valider une clé candidate
target_prefix = np.array([ord(c) for c in "BEGIN:"], dtype=np.uint8)

print("Texte original  :", texte_original)
print("Clé secrète     :", cle_secrete_texte)
print("Texte chiffré   :", [ord(c) for c in ciphertext_str[:10]], "...")

Texte original  : BEGIN: rapport_confidentiel_2024.txt
Clé secrète     : STORM
Texte chiffré   : [17, 17, 8, 27, 3, 105, 116, 61, 51, 61] ...


## Cellule 2 — Version CPU intelligente : lettre par lettre

Code adapté depuis : https://gist.github.com/mgeeky/589b2cf781901288dfea0894a780ff98  
(Mariusz B., 2016)

L'idée : pour retrouver la lettre N de la clé, on regarde uniquement les positions
du texte chiffré qui correspondent à cette lettre (positions N, N+5, N+10...).
On teste les 256 valeurs ASCII possibles jusqu'à trouver celle qui donne le bon texte.

**Limite** : nécessite de connaître le texte original en entier.

In [ ]:
def xorstring(s, k):
    out = [0 for c in range(len(s))]
    key = []
    if type(k) == type(int):
        key = [k,]
    else:
        key = [ki for ki in k]

    for i in range(len(key)):
        for j in range(i, len(s), len(key)):
            out[j] = chr(ord(s[j]) ^ key[i])

    return ''.join(out)


def brute(input_xored, expected_output, key_len):
    key = []

    if len(input_xored) != len(expected_output):
        print('[!] Input xored and expected output lengths not match!')
        return False

    for i in range(key_len):
        cipher_letters    = [input_xored[x]    for x in range(i, len(input_xored), key_len)]
        plaintext_letters = [expected_output[x] for x in range(i, len(input_xored), key_len)]

        found = False
        for k in range(256):
            found = True
            for j in range(len(cipher_letters)):
                if chr(ord(cipher_letters[j]) ^ k) != plaintext_letters[j]:
                    found = False
                    break

            if found:
                key.append(k)
                break
            found = False

        if not found:
            print('[!] Could not find partial key value.')
            break

    return key, xorstring(input_xored, key) == expected_output


print("Lancement récupération de clé CPU (lettre par lettre)...")
start_time = time.time()
key, status = brute(ciphertext_str, texte_original, len(cle_secrete_texte))
cpu_smart_time = time.time() - start_time

if status:
    found_key_cpu_smart = "".join([chr(k) for k in key])
    print(f"Clé trouvée : {found_key_cpu_smart}")
else:
    print("Clé non trouvée")

print(f"Temps      : {cpu_smart_time:.6f} secondes")
print()
print("=> Très rapide, mais nécessite le texte original complet.")
print("   Dans un vrai scénario d'attaque on ne l'a pas — on passe au brute-force.")

Lancement récupération de clé CPU (lettre par lettre)...
Clé trouvée : STORM
Temps      : 0.000130 secondes

=> Très rapide, mais nécessite le texte original complet.
   Dans un vrai scénario d'attaque on ne l'a pas — on passe au brute-force.


## Cellule 3 — Brute-force séquentiel sur CPU

Nouveau scénario : on ne connaît que le préfixe `"BEGIN:"`.
On teste toutes les $26^5 = 11\,881\,376$ combinaisons une par une.
C'est le même algorithme que le GPU — mais exécuté séquentiellement.

In [ ]:
# fonction qui cherche la clé en testant toutes les combinaisons possibles
def crack_cpu_bruteforce(ciphertext, target_prefix, key_length=5):

    # nombre total de clés à tester : 26^5 = 11,8 m
    total_keys = 26 ** key_length

    # on teste chaque clé une par une, dans l'ordre
    for tid in range(total_keys):

        # 1ère lettre de la clé : on prend le reste de tid divisé par 26
        # (0→'A', 1→'B', ..., 25→'Z', puis ça repart à 'A')
        k0 = (tid % 26) + 65

        # 2ème lettre : on décale d'un rang en divisant par 26
        k1 = ((tid // 26) % 26) + 65

        # 3ème, 4ème, 5ème lettres : même principe, on monte d'un rang à chaque fois
        k2 = ((tid // (26**2)) % 26) + 65
        k3 = ((tid // (26**3)) % 26) + 65
        k4 = ((tid // (26**4)) % 26) + 65

        # on assemble les 5 lettres pour former la clé
        key = [k0, k1, k2, k3, k4]

        # on suppose que la clé est correcte jusqu'à preuve du contraire
        match = True

        # on vérifie la clé sur chaque caractère du préfixe
        for i in range(len(target_prefix)):

            # si le déchiffrement XOR ne correspond pas au préfixe attendu, mauvaise clé
            if (ciphertext[i] ^ key[i % key_length]) != target_prefix[i]:
                match = False
                # on arrête dès le premier caractère qui ne correspond pas, pour aller plus vite
                break

        # clé trouvée : on convertit les codes ASCII en lettres et on retourne le résultat
        if match:
            return "".join([chr(k) for k in key])

    # aucune clé trouvée parmi les 11,8 millions testées
    return "Non trouvé"


print("Lancement brute-force CPU (11,8 millions de clés)... patience")
# on mesure le temps d'exécution
start_time = time.time()
found_key_cpu_brute = crack_cpu_bruteforce(ciphertext_np, target_prefix)
cpu_brute_time = time.time() - start_time

print(f"Clé trouvée : {found_key_cpu_brute}")
print(f"Temps      : {cpu_brute_time:.4f} secondes")

Lancement brute-force CPU (11,8 millions de clés)... patience
Clé trouvée : STORM
Temps      : 3.3948 secondes


## Cellule 4 — Kernel CUDA : brute-force parallèle

Même algorithme que la cellule 3, mais **1 thread = 1 clé**.
On lance 11,8 millions de threads en simultané au lieu d'une boucle séquentielle.

In [ ]:
@cuda.jit
def crack_kernel_cuda(ciphertext, target_prefix, result_array):

    # chaque thread reçoit un numéro unique, c'est son identifiant dans la grille
    tid = cuda.grid(1)

    # on s'assure de ne pas dépasser le nombre total de clés
    if tid < (26**5):

        # 1ère lettre de la clé : on prend le reste de tid divisé par 26
        k0 = (tid % 26) + 65

        # 2ème lettre : on décale d'un rang en divisant par 26
        k1 = ((tid // 26) % 26) + 65

        # 3ème, 4ème, 5ème lettres : même principe, on monte d'un rang à chaque fois
        k2 = ((tid // (26**2)) % 26) + 65
        k3 = ((tid // (26**3)) % 26) + 65
        k4 = ((tid // (26**4)) % 26) + 65

        # en CUDA les tableaux locaux doivent avoir une taille fixe connue à l'avance
        key = cuda.local.array(5, dtype=np.uint8)
        key[0] = k0
        key[1] = k1
        key[2] = k2
        key[3] = k3
        key[4] = k4

        # on suppose que la clé est correcte jusqu'à preuve du contraire
        match = True

        # on vérifie la clé sur chaque caractère du préfixe
        for i in range(len(target_prefix)):

            # si le déchiffrement XOR ne correspond pas au préfixe attendu, mauvaise clé
            if (ciphertext[i] ^ key[i % 5]) != target_prefix[i]:
                match = False
                # on arrête dès le premier caractère qui ne correspond pas
                break

        # si ce thread a trouvé la bonne clé, il sauvegarde son numéro dans le résultat
        if match:
            result_array[0] = tid

## Cellule 5 — Lancement GPU et comparaison finale

In [ ]:
total_keys      = 26**5
threadsperblock = 256
blockspergrid   = math.ceil(total_keys / threadsperblock)

print(f"Nombre de blocs  : {blockspergrid}")
print(f"Threads par bloc : {threadsperblock}")
print(f"Total threads    : {blockspergrid * threadsperblock}")

# transfert des données depuis la RAM vers la mémoire du GPU
d_ciphertext    = cuda.to_device(ciphertext_np)
d_target_prefix = cuda.to_device(target_prefix)
d_result        = cuda.to_device(np.array([-1], dtype=np.int64))


# on mesure le temps d'exécution
start_time = time.time()
crack_kernel_cuda[blockspergrid, threadsperblock](d_ciphertext, d_target_prefix, d_result)
# on attend que tous les threads aient fini avant de lire le résultat
cuda.synchronize()
gpu_time = time.time() - start_time

# récupération du numéro du thread gagnant depuis la mémoire GPU vers la RAM
winner_tid = d_result.copy_to_host()[0]

# reconversion du numéro en clé lisible (mêmes modulos qu'à l'envoi)
k0 = chr((winner_tid % 26) + 65)
k1 = chr(((winner_tid // 26) % 26) + 65)
k2 = chr(((winner_tid // (26**2)) % 26) + 65)
k3 = chr(((winner_tid // (26**3)) % 26) + 65)
k4 = chr(((winner_tid // (26**4)) % 26) + 65)
found_key_gpu = f"{k0}{k1}{k2}{k3}{k4}"

print(f"\n{'='*55}")
print(f"RÉSULTATS")
print(f"{'='*55}")
print(f"Clé réelle                              : {cle_secrete_texte}")
print(f"Clé trouvée CPU intelligent              : {found_key_cpu_smart}")
print(f"Clé trouvée CPU brute-force             : {found_key_cpu_brute}")
print(f"Clé trouvée GPU brute-force             : {found_key_gpu}")
print(f"{'='*55}")
print(f"Temps CPU intelligent (texte complet)   : {cpu_smart_time:.6f} s")
print(f"Temps CPU brute-force (préfixe seul)    : {cpu_brute_time:.4f} s")
print(f"Temps GPU brute-force (préfixe seul)    : {gpu_time:.4f} s")
print(f"{'='*55}")
print(f"Speedup GPU vs CPU brute-force          : x{cpu_brute_time / gpu_time:.1f}")

Nombre de blocs  : 46412
Threads par bloc : 256
Total threads    : 11881472

RÉSULTATS
Clé réelle                              : STORM
Clé trouvée CPU intelligent              : STORM
Clé trouvée CPU brute-force             : STORM
Clé trouvée GPU brute-force             : STORM
Temps CPU intelligent (texte complet)   : 0.000130 s
Temps CPU brute-force (préfixe seul)    : 3.3948 s
Temps GPU brute-force (préfixe seul)    : 0.0006 s
Speedup GPU vs CPU brute-force          : x5835.6
